# NCERT Preprocessing Pipeline — v4
## Zero hardcoding — all errors detected automatically from data structure

### Errors fixed (from both Hindi and English annotation reviews)

| # | Error | Language | Auto-detection method |
|---|---|---|---|
| 1 | Header injected mid-sentence | English | Word-level extraction sorted by (y, x) |
| 2 | Multi-column TOC merging | English | Large internal gap detection (3+ spaces) |
| 3 | Running headers / page labels | Both | Multi-signal: frequency + edge position |
| 4 | Pedagogical blocks (LET'S EXPLORE) | English | ALL-CAPS ratio ≥ 80% + short line |
| 5 | Figure caption split across blocks | Both | Short fragment (<15 chars, no terminal punct) merger |
| 6 | Math formula linearized (15 24 = 360) | English | N N = N pattern detector — flagged and dropped |
| 7 | Syllable repetition (प्राप्राकृतिक) | Hindi | Repeating 2–6 char substring finder |
| 8 | Char duplication (आपप → आप) | Hindi | Safe consecutive-char dedup (geminates preserved) |
| 9 | Ligature/matra decomposition (विज्ान) | Hindi | Halant-orphan detector — flags for review |
| 10 | ZWJ (U+200D) wrongly removed | Hindi | ZWJ explicitly excluded from cleaner |
| 11 | PDF line-wrap broken sentences | Both | Continuation rejoiner + English dehyphenation |
| 12 | Out-of-range Unicode tokens | Both | Per-language 70% char-validity filter |

In [ ]:
import fitz  # PyMuPDF
import re
import unicodedata
import json
from collections import Counter, defaultdict
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize

---
## STEP 1 — Word-Level PDF Extraction (best reading order)

**Why word-level, not block-level or raw text:**

| Method | Problem |
|---|---|
| `get_text("text")` | Returns text in PDF stream order — loses visual layout entirely |
| `get_text("blocks")` | Better, but blocks can still span multiple columns |
| `get_text("words")` ✅ | Word-level bounding boxes → sort by `(y_band, x0)` → perfect reading order |

**How it works:**
Each word has coordinates `(x0, y0, x1, y1)`. Words at the same `y0` (±8px band) are on the same visual line. Sorting by `x0` within the band gives left→right order. This **automatically eliminates column-mixing** — columns are at different x0, so they sort into their own streams naturally.

This fixes the root cause of: *header injected mid-paragraph*, *multi-column TOC merging*, *reading order lost*.

In [ ]:
def reconstruct_lines_from_words(words, y_band=8):
    """
    Reconstruct visual text lines from word-level extraction.

    Groups words within the same y_band (pixels) into one line,
    then sorts left→right by x0 within each band.

    Automatically handles:
    - Multi-column layouts (columns have different x0, stay separate)
    - Slight y-misalignments between words on the same visual line
    - Header blocks that visually separate from paragraph blocks

    No hardcoding. Works on any PDF layout.

    word tuple: (x0, y0, x1, y1, text, block_no, line_no, word_no)
    """
    if not words:
        return []

    # Group words by y-band
    bands = defaultdict(list)
    for w in words:
        band_key = round(w[1] / y_band) * y_band
        bands[band_key].append(w)

    lines = []
    for band_key in sorted(bands.keys()):        # top → bottom
        row = sorted(bands[band_key], key=lambda w: w[0])  # left → right
        line_text = ' '.join(w[4] for w in row).strip()
        if line_text:
            lines.append(line_text)
    return lines


def extract_pages(pdf_path, y_band=8):
    """
    Word-level page extraction with correct reading order.
    Returns list of lists — each inner list = lines on one page.

    Uses get_text('words') → sort by (y_band, x0).
    NFC normalization applied per word.
    """
    doc = fitz.open(pdf_path)
    page_lines = []

    for page in doc:
        words = page.get_text("words")  # (x0,y0,x1,y1,text,block,line,word)
        # NFC normalize each word
        words = [
            (w[0], w[1], w[2], w[3], unicodedata.normalize("NFC", w[4]), w[5], w[6], w[7])
            for w in words if w[4].strip()
        ]
        lines = reconstruct_lines_from_words(words, y_band=y_band)
        page_lines.append(lines)

    return page_lines

---
## STEP 2 — Automatic Noise Detection (zero hardcoding)

No book titles, chapter names, language-specific words, or regex patterns.
Everything derived from **how lines behave across pages**.

| Signal | Threshold | What it catches |
|---|---|---|
| A — Page frequency | ≥ 40% of pages | Book title / chapter header repeating every page |
| B — Edge position | ≥ 75% of appearances at top or bottom 15% of page | Running headers and footers |
| C — Pure numeric | Only digits/spaces, length ≤ 6 | Page numbers — Latin (1,2,3) or Devanagari (१,२,३) |
| D — Short + repeated | Length ≤ 25 chars AND on ≥ 50% of pages | Unit labels, section numbers |

**Rule:** Signal C alone → noise (page numbers are unambiguous).  
Any **2 or more** of (A, B, D) → noise.  
Single signal only → keep (prevents real repeated sentences from being removed).

In [ ]:
def detect_noise_auto(page_lines, top_n=30):
    """
    Fully automatic noise detection using multi-signal evidence.
    No hardcoded strings. Works on any language, any publisher.

    Returns:
        noise_set      — set of line strings to remove
        noise_evidence — dict of line -> list of reasons
        freq           — Counter of all lines
    """
    total_pages    = len(page_lines)
    all_lines      = [l for page in page_lines for l in page if l.strip()]
    freq           = Counter(all_lines)
    line_pages     = defaultdict(set)
    line_positions = defaultdict(list)

    for page_idx, page in enumerate(page_lines):
        clean = [l for l in page if l.strip()]
        for pos_idx, line in enumerate(clean):
            line_pages[line].add(page_idx)
            pos_frac = pos_idx / max(len(clean) - 1, 1)
            line_positions[line].append(pos_frac)

    noise_evidence = {}

    for line in freq:
        stripped   = line.strip()
        line_len   = len(stripped)
        page_ratio = len(line_pages[line]) / total_pages
        positions  = line_positions[line]
        edge_count = sum(1 for p in positions if p < 0.15 or p > 0.85)
        edge_score = edge_count / len(positions) if positions else 0

        # Four independent signals
        sig_A = page_ratio >= 0.40
        sig_B = edge_score >= 0.75 and freq[line] >= 2
        sig_C = (bool(re.fullmatch(r'[\d\s\u0966-\u096F\.\-\(\)]+', stripped))
                 and line_len <= 6)
        sig_D = line_len <= 25 and freq[line] >= max(3, int(total_pages * 0.50))

        reasons = []
        if sig_C:
            reasons.append("page_number")
        elif sum([sig_A, sig_B, sig_D]) >= 2:
            if sig_A: reasons.append(f"high_freq ({page_ratio:.0%} of pages)")
            if sig_B: reasons.append(f"always_at_edge (score={edge_score:.2f})")
            if sig_D: reasons.append(f"short_repeated ({line_len}ch, {freq[line]}x)")

        if reasons:
            noise_evidence[line] = reasons

    noise_set = set(noise_evidence.keys())

    # --- Report ---
    print(f"\n{'='*68}")
    print(" NOISE DETECTION REPORT")
    print(f"{'='*68}")
    print(f" Pages: {total_pages}  |  Unique lines: {len(freq)}  |  Flagged noise: {len(noise_set)}")
    print(f" {'FREQ':>5}  {'PG%':>5}  {'EDGE':>5}  {'STATUS':<10}  LINE")
    print(f" {'-'*60}")
    for line, count in freq.most_common(top_n):
        pr = len(line_pages[line]) / total_pages
        ps = line_positions[line]
        es = sum(1 for p in ps if p < 0.15 or p > 0.85) / len(ps) if ps else 0
        st = "[NOISE]" if line in noise_set else "keep"
        print(f" {count:>5}  {pr:>4.0%}  {es:>4.0%}  {st:<10}  {repr(line[:38])}")
    print('='*68)

    return noise_set, noise_evidence, freq


def apply_noise_filter(page_lines, noise_set, header_rows=1, footer_rows=1):
    """
    Remove detected noise lines.
    Also strips the first header_rows and last footer_rows lines per page
    as a positional safety net.
    """
    filtered = []
    for page in page_lines:
        clean = [l for l in page if l.strip()]
        body  = (clean[header_rows : len(clean) - footer_rows]
                 if len(clean) > header_rows + footer_rows else clean)
        filtered.extend(line for line in body if line not in noise_set)
    return filtered

---
## STEP 3 — Structural Line Filters (ALL-CAPS, column-merge, math)

Three new filters, all purely structural — no hardcoded words:

### 3a — ALL-CAPS pedagogical blocks
Lines like `LET'S EXPLORE`, `DON'T MISS OUT`, `QUESTIONS AND ACTIVITIES`.
Detected by: uppercase ratio ≥ 80% + length ≤ 50 chars.
These are textbook navigation labels — not narrative text, drop them.

### 3b — Multi-column merge detector
When two column headings land on the same visual line: `Family and Community   Governance Local Government`.
Detected by: 3+ consecutive spaces in middle of a line of 5+ words.
Word-level extraction reduces this, but not always eliminates it.

### 3c — Linearized math expression detector  
`15 24 = 360` — a formula where the operator was lost.
We cannot recover the operator. Detected by: `digit... digit... =... digit` pattern, short line.
Safest to drop — these add noise to NLP training data.

In [ ]:
# ------- 3a: ALL-CAPS pedagogical block -------
def caps_ratio(line):
    """Fraction of alphabetic chars that are uppercase."""
    alpha = [c for c in line if c.isalpha()]
    return sum(1 for c in alpha if c.isupper()) / len(alpha) if alpha else 0

def is_pedagogical_block(line):
    """
    Detect ALL-CAPS instructional labels.
    Structural rule: caps_ratio >= 0.80 AND len <= 50.
    No hardcoded keywords. Works for any textbook publisher.
    """
    stripped = line.strip()
    return (
        3 < len(stripped) <= 50 and
        caps_ratio(stripped) >= 0.80
    )


# ------- 3b: Multi-column merge detector -------
def is_column_merged(line):
    """
    Detect a line that is two column texts merged into one.
    Structural signal: 3+ consecutive internal spaces + enough words.
    Word-level extraction mostly prevents this, but catches any remainder.
    """
    return bool(re.search(r'\S {3,}\S', line)) and len(line.split()) > 4


# ------- 3c: Linearized math expression -------
def is_math_fragment(line):
    """
    Detect formulas where the operator was lost: '15 24 = 360'.
    Cannot recover the operator — safest to drop.
    Pattern: multiple numeric tokens + '=' + short line.
    """
    tokens = line.strip().split()
    if len(tokens) < 3 or len(line) > 80:
        return False
    has_eq  = '=' in tokens
    num_cnt = sum(1 for t in tokens if re.fullmatch(r'[\d°\./\-\+]+', t))
    return has_eq and num_cnt >= 2


def apply_structural_filters(lines):
    """
    Apply all three structural filters in sequence.
    Returns: (kept_lines, drop_report)
    """
    kept        = []
    drop_report = defaultdict(list)

    for line in lines:
        if is_pedagogical_block(line):
            drop_report['pedagogical'].append(line)
        elif is_column_merged(line):
            drop_report['column_merged'].append(line)
        elif is_math_fragment(line):
            drop_report['math_fragment'].append(line)
        else:
            kept.append(line)

    print(f"  [Structural filters]")
    for reason, dropped in drop_report.items():
        print(f"    {reason}: {len(dropped)} lines dropped")
        for l in dropped[:3]:  # show up to 3 examples
            print(f"      e.g. {repr(l[:70])}")

    return kept

---
## STEP 4 — Hindi PDF Glyph Error Correction

Three error types — two are auto-fixed, one is auto-detected and flagged.

### Error 1 — Syllable repetition (glyph overlap)
PDF renderer places glyphs at overlapping positions; PyMuPDF reads each twice.
```
प्राप्राकृतिक → प्राकृतिक
```
**Fix:** Find any substring of 2–6 chars immediately repeated → keep one.
Purely structural — no dictionary, works on any script.

### Error 2 — Character duplication (glyph code collision)
Two glyph IDs map to the same Unicode codepoint.
```
आपप → आप     पपृथवी → पृथवी
```
**Fix:** Remove consecutive identical chars — but preserve genuine geminates:
- After halant: `त्त`, `न्न` (keep)
- Before halant: `कक्` in `कक्षा` (keep — second क is part of the conjunct)

### Error 3 — Ligature/matra decomposition (FLAGGED, not auto-fixed)
PDF ligatures `ज्ञ`, `ध्य`, `त्य` split into halant + wrong char.
```
विज्ान → विज्ञान   अधययन → अध्ययन
```
Cannot auto-fix — the missing consonant varies per word.  
Detected by: **halant orphan analysis** — halant (्) not followed by a valid consonant.
These are reported in the quality report for manual review.

In [ ]:
HALANT              = '\u094D'  # ् virama
VALID_AFTER_HALANT  = set(chr(c) for c in range(0x0915, 0x093A)) | {'\u200D'}
# Devanagari consonants क (0x0915) to ह (0x0939), plus ZWJ


def fix_syllable_repetition(text):
    """
    Remove immediately repeated substrings of length 2–6 chars.
    Catches glyph-overlap duplication during PDF extraction.
    Ex: प्राप्राकृतिक → प्राकृतिक
    Purely structural — no dictionary, works on any script.
    """
    for length in range(6, 1, -1):
        text = re.sub(r'(.{' + str(length) + r'})\1+', r'\1', text)
    return text


def fix_char_duplication(text):
    """
    Remove consecutive identical chars — but keep:
    - Chars AFTER halant (genuine geminates: त्त, न्न)
    - Chars BEFORE halant (part of upcoming conjunct: कक् in कक्षा)
    Ex: आपप → आप
    """
    chars  = list(text)
    n      = len(chars)
    result = []
    i      = 0
    while i < n:
        ch = chars[i]
        if result and ch == result[-1] and ch != HALANT:
            prev_prev = result[-2] if len(result) >= 2 else ''
            next_ch   = chars[i + 1] if i + 1 < n else ''
            if prev_prev == HALANT or next_ch == HALANT:
                result.append(ch)  # safe geminate — keep
            # else: duplication artifact — skip
        else:
            result.append(ch)
        i += 1
    return ''.join(result)


def find_broken_conjuncts(text):
    """
    Detect halant (्) NOT followed by a valid Devanagari consonant.
    This indicates a broken ligature (ज्ञ split into ज् + wrong char).
    Returns list of (position, context_string) for review.
    Cannot auto-fix — missing consonant varies per word.
    """
    broken = []
    for i, ch in enumerate(text):
        if ch == HALANT:
            nxt = text[i + 1] if i + 1 < len(text) else ''
            if nxt not in VALID_AFTER_HALANT:
                broken.append((i, text[max(0, i - 4):i + 5]))
    return broken


def fix_hindi_pdf_errors(text):
    """
    Automatic correction for Hindi PDF glyph extraction errors.
    Order: NFC → syllable dedup → char dedup.
    """
    text = unicodedata.normalize("NFC", text)
    text = fix_syllable_repetition(text)
    text = fix_char_duplication(text)
    return text


# --- Verification ---
print("Hindi glyph fix verification:")
tests = [
    ("आपप",           "आप"),
    ("पपृथवी",        "पृथवी"),
    ("प्राप्राकृतिक", "प्राकृतिक"),
    ("प्राप्राप्त",   "प्राप्त"),
    ("अत्यंत",        "अत्यंत"),   # correct word — must be unchanged
    ("विज्ञान",       "विज्ञान"),  # correct word — must be unchanged
    ("कक्षा",         "कक्षा"),    # second क before halant — preserve
    ("अत्त",          "अत्त"),     # genuine geminate after halant — preserve
]
for inp, expected in tests:
    out = fix_hindi_pdf_errors(inp)
    print(f"  '{inp}' → '{out}'  {'✓' if out == expected else f'✗ expected {expected}'}")

---
## STEP 5 — Invisible Character Cleaner (ZWJ preserved)

`U+200D` ZWJ (Zero Width Joiner) is **kept** — it controls Devanagari ligature formation and is linguistically meaningful in Hindi.  
Removed: ZWS (U+200B), LRM/RLM marks, BOM (U+FEFF), directional overrides, soft-hyphen.

In [ ]:
GARBAGE_INVISIBLE_RE = re.compile(
    r'[\u200B'   # Zero Width Space
    r'\u200E'    # Left-to-Right Mark
    r'\u200F'    # Right-to-Left Mark
    r'\u00AD'    # Soft Hyphen
    r'\uFEFF'    # BOM / Zero Width No-Break Space
    r'\u202A-\u202E]'  # Directional formatting overrides
    # U+200D (ZWJ) intentionally NOT listed — needed for Devanagari ligatures
    # U+200C (ZWNJ) intentionally NOT listed — used in some Indic contexts
)

def clean_invisible_chars(text):
    return GARBAGE_INVISIBLE_RE.sub('', text)

---
## STEP 6 — Fragment Merger (figure captions, split labels)

Word-level extraction sometimes puts a block label (`Fig. 1.1`) as a standalone very short line before the caption text.

**Auto-detection:** Line is a fragment if:
- Length < 15 chars, AND  
- Does NOT end with sentence-terminating punctuation (`। . ? ! : ;`)

No hardcoded words like `Fig`, `Figure`, `चित्र` — purely structural.

In [ ]:
def merge_fragments(lines):
    """
    Merge very short incomplete lines with the line that follows.
    Fragment criteria: len < 15 AND no sentence-ending punctuation.
    Catches: figure captions, split section labels, partial headings.
    No hardcoded patterns — purely structural.
    """
    merged = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        is_fragment = (
            0 < len(line) < 15 and
            not re.search(r'[।.?!:;]\s*$', line)
        )
        if is_fragment and i + 1 < len(lines):
            merged.append(line + ' ' + lines[i + 1].strip())
            i += 2
        else:
            merged.append(line)
            i += 1
    return [l for l in merged if l.strip()]

---
## STEP 7 — PDF Line-Wrap Rejoining

Even with word-level extraction, sentence boundaries set by print layout remain.
A line not ending with sentence-terminating punctuation is a continuation of the previous.
English also handles hyphenated word-breaks (`develop-` + `ment` → `development`).

In [ ]:
SENT_END_RE = re.compile(r'[।?!\.\"\']\s*$')

def rejoin_wrapped_lines(lines, lang="hindi"):
    """
    Rejoin lines that are print-layout continuations of the previous line.
    A line NOT ending with sentence punctuation is merged with the next.
    English: also dehyphenates word-breaks (line ends with '-').
    """
    joined = []
    buffer = ""
    for line in lines:
        line = line.strip()
        if not line:
            if buffer:
                joined.append(buffer)
            buffer = ""
            continue
        if lang == "english" and buffer.endswith('-'):
            buffer = buffer[:-1] + line   # dehyphenate
        else:
            buffer = (buffer + ' ' + line).strip() if buffer else line
        if SENT_END_RE.search(line):
            joined.append(buffer)
            buffer = ""
    if buffer:
        joined.append(buffer)
    return joined

---
## STEP 8 — Unicode Validation (per-language)

Each language only allows its own script's Unicode ranges.
Token dropped if < 70% of its chars are in the valid range for that language.
Line dropped if all tokens are invalid.

In [ ]:
def is_valid_hindi_char(ch):
    cp = ord(ch)
    return (
        0x0900 <= cp <= 0x097F    # Devanagari block
        or cp == 0x200D            # ZWJ — preserve
        or cp in (0x0964, 0x0965)  # Danda (।), Double Danda (॥)
        or 0xA8E0 <= cp <= 0xA8FF  # Devanagari Extended
        or ch.isspace()
        or ch.isdigit()
        or 0x0020 <= cp <= 0x0040  # space + basic punctuation
        or cp == 0x002D            # hyphen
        or 0x2018 <= cp <= 0x201F  # typographic quotes
    )

def is_valid_english_char(ch):
    return 0x0020 <= ord(ch) <= 0x007E  # printable ASCII

def unicode_validate_lines(lines, lang="hindi"):
    """
    Drop tokens where < 70% of chars are in valid range for the language.
    Drop entire line if no valid tokens remain.
    """
    checker = is_valid_hindi_char if lang == "hindi" else is_valid_english_char
    result, dropped = [], 0
    for line in lines:
        valid_tokens = [
            t for t in line.split()
            if t and sum(1 for c in t if checker(c)) / len(t) >= 0.70
        ]
        cleaned = ' '.join(valid_tokens)
        if cleaned.strip():
            result.append(cleaned)
        else:
            dropped += 1
    print(f"  [Unicode - {lang}] Lines dropped entirely: {dropped}")
    return result

---
## STEP 9 — Remaining Cleaners

In [ ]:
def remove_numeric_only(lines):
    """Drop lines that are purely numeric — catches any leftover page numbers."""
    return [l for l in lines
            if not re.fullmatch(r'[\d\s\u0966-\u096F\.\-\(\)]+', l.strip())]

def devanagari_ratio(line):
    total = len(line)
    return sum(1 for c in line if '\u0900' <= c <= '\u097F') / total if total else 0

def normalize_spacing(text, lang="english"):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.!?])', r'\1', text)
    text = re.sub(r'([,.!?])([^\s])', r'\1 \2', text)
    if lang == "hindi":
        text = re.sub(r'\s*।\s*', '। ', text)
    return text.strip()

def segment_sentences(lines, lang="english"):
    sentences = []
    for line in lines:
        if lang == "english":
            sentences.extend(sent_tokenize(line))
        else:
            sentences.extend(re.split(r'(?<=[।?!])\s+', line))
    return [s.strip() for s in sentences if len(s.strip()) >= 20]

---
## STEP 10 — Full Pipelines

Both pipelines now share the same upgraded extraction (word-level), noise detector, and structural filters.  
Hindi additionally runs glyph correction. English skips Devanagari ratio filter.

In [ ]:
def preprocess_hindi(pdf_path, min_dev_ratio=0.4):
    """
    Hindi pipeline:
      1.  Word-level extraction — correct reading order (no column mixing)
      2.  Auto noise detection — frequency + position signals
      3.  Structural filters  — pedagogical blocks, column merges, math
      4.  Invisible char clean — ZWJ preserved
      5.  Fragment merger     — figure captions, split labels
      6.  Line-wrap rejoining
      7.  Hindi glyph correction — syllable dedup, char dedup
      8.  Unicode validation  — Devanagari range
      9.  Numeric + Devanagari ratio filter
      10. Spacing normalization
      11. Sentence segmentation on danda
    """
    print(f"\n{'#'*68}")
    print(f" HINDI: {pdf_path}")
    print(f"{'#'*68}")

    # Step 1 — Word-level extraction
    page_lines = extract_pages(pdf_path)
    print(f" Pages extracted: {len(page_lines)}")
    total_raw = sum(len(p) for p in page_lines)
    print(f" Total lines raw: {total_raw}")

    # Step 2 — Auto noise detection
    noise_set, _, _ = detect_noise_auto(page_lines)
    lines = apply_noise_filter(page_lines, noise_set)
    print(f" After noise filter     : {len(lines)} lines")

    # Step 3 — Structural filters
    lines = apply_structural_filters(lines)

    # Steps 4–6 — Invisible chars, fragments, line-wrap
    lines = [clean_invisible_chars(l) for l in lines]
    lines = [l for l in lines if l.strip()]
    lines = merge_fragments(lines)
    lines = rejoin_wrapped_lines(lines, lang="hindi")
    print(f" After rejoining        : {len(lines)} lines")

    # Step 7 — Glyph correction (Hindi-specific)
    lines = [fix_hindi_pdf_errors(l) for l in lines]

    # Steps 8–10 — Validation and normalization
    lines = unicode_validate_lines(lines, lang="hindi")
    lines = remove_numeric_only(lines)

    before = len(lines)
    lines  = [l for l in lines if devanagari_ratio(l) >= min_dev_ratio]
    print(f" After Devanagari filter: {len(lines)} lines (dropped {before - len(lines)})")

    lines = [normalize_spacing(l, lang="hindi") for l in lines]
    lines = [l for l in lines if len(l) > 25]

    # Step 11 — Segmentation
    sentences = segment_sentences(lines, lang="hindi")
    print(f" Final Hindi sentences  : {len(sentences)}")
    return sentences


def preprocess_english(pdf_path):
    """
    English pipeline:
      1.  Word-level extraction — correct reading order (no column mixing)
      2.  Auto noise detection — frequency + position signals
      3.  Structural filters  — pedagogical blocks, column merges, math
      4.  Invisible char clean
      5.  Fragment merger     — figure captions, split labels
      6.  Line-wrap rejoining + dehyphenation
      7.  Unicode validation  — ASCII range
      8.  Numeric filter
      9.  Spacing normalization
      10. NLTK sentence segmentation
    """
    print(f"\n{'#'*68}")
    print(f" ENGLISH: {pdf_path}")
    print(f"{'#'*68}")

    # Step 1 — Word-level extraction
    page_lines = extract_pages(pdf_path)
    print(f" Pages extracted: {len(page_lines)}")
    total_raw = sum(len(p) for p in page_lines)
    print(f" Total lines raw: {total_raw}")

    # Step 2 — Auto noise detection
    noise_set, _, _ = detect_noise_auto(page_lines)
    lines = apply_noise_filter(page_lines, noise_set)
    print(f" After noise filter     : {len(lines)} lines")

    # Step 3 — Structural filters
    lines = apply_structural_filters(lines)

    # Steps 4–6 — Invisible chars, fragments, line-wrap
    lines = [clean_invisible_chars(l) for l in lines]
    lines = [l for l in lines if l.strip()]
    lines = merge_fragments(lines)
    lines = rejoin_wrapped_lines(lines, lang="english")
    print(f" After rejoining        : {len(lines)} lines")

    # Steps 7–9 — Validation and normalization
    lines = unicode_validate_lines(lines, lang="english")
    lines = remove_numeric_only(lines)
    lines = [normalize_spacing(l, lang="english") for l in lines]
    lines = [l for l in lines if len(l) > 25]

    # Step 10 — Segmentation
    sentences = segment_sentences(lines, lang="english")
    print(f" Final English sentences: {len(sentences)}")
    return sentences


# ======================== RUN ========================
hindi_sentences   = preprocess_hindi("ncert_6_social_hindi.pdf")
english_sentences = preprocess_english("ncert_6_social.pdf")

with open("hindi_sentences_v4.txt",   "w", encoding="utf-8") as f:
    f.writelines(s + "\n" for s in hindi_sentences)
with open("english_sentences_v4.txt", "w", encoding="utf-8") as f:
    f.writelines(s + "\n" for s in english_sentences)

print(f"\n Saved {len(hindi_sentences)} Hindi   → hindi_sentences_v4.txt")
print(f" Saved {len(english_sentences)} English → english_sentences_v4.txt")

---
## STEP 11 — Quality Report

Automatically checks output for:
- **Latin-1 chars in Hindi output** — residual Æ-type noise that slipped through
- **Broken conjuncts** — halant (्) not followed by valid consonant → need manual review
- **Sentence length stats** — catches very short/long outliers
- **Sample sentences** — visual inspection

In [ ]:
def quality_report(sentences, lang, n_sample=15):
    print(f"\n{'='*68}")
    print(f" QUALITY REPORT: {lang.upper()}")
    print(f"{'='*68}")

    if not sentences:
        print(" No sentences — check pipeline!")
        return

    lengths = [len(s) for s in sentences]
    print(f" Total sentences  : {len(sentences)}")
    print(f" Length — min:{min(lengths)}  max:{max(lengths)}  avg:{sum(lengths)/len(lengths):.0f}")

    # --- Automatic issue detection ---
    latin1_hits, broken_total, flagged = 0, 0, []

    for s in sentences:
        issues = []

        if lang == "hindi":
            # Residual Latin-1 chars (likely noise artifacts like Æ)
            l1 = [repr(ch) for ch in s if 0x0080 <= ord(ch) <= 0x00FF]
            if l1:
                latin1_hits += 1
                issues.append(f"Latin-1 chars: {l1}")

            # Broken conjuncts (halant orphans)
            bc = find_broken_conjuncts(s)
            if bc:
                broken_total += len(bc)
                issues.append(f"Broken conjuncts at: {[ctx for _, ctx in bc]}")

        if issues:
            flagged.append((s, issues))

    if lang == "hindi":
        print(f" Latin-1 in Hindi  : {latin1_hits} sentences  {'✓ none' if latin1_hits == 0 else '⚠ needs review'}")
        print(f" Broken conjuncts  : {broken_total} total  {'✓ none' if broken_total == 0 else '⚠ manual fix needed'}")

    print(f"\n First {n_sample} sentences:")
    for i, s in enumerate(sentences[:n_sample]):
        print(f"  [{i+1:3}] {s}")

    if flagged:
        print(f"\n ⚠  FLAGGED ({len(flagged)} sentences need manual review):")
        for s, issues in flagged[:10]:
            print(f"   {s[:80]}")
            for iss in issues:
                print(f"     → {iss}")
    else:
        print(f"\n ✓ No issues detected automatically.")


quality_report(hindi_sentences,   lang="hindi",   n_sample=15)
quality_report(english_sentences, lang="english", n_sample=15)

---
## STEP 12 — Sentence Alignment (LaBSE)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('LaBSE')

def get_sentence_alignments(hindi_sents, english_sents, top_k=4):
    print(" Encoding Hindi sentences...")
    hi_emb = model.encode(hindi_sents,   show_progress_bar=True, normalize_embeddings=True)
    print(" Encoding English sentences...")
    en_emb = model.encode(english_sents, show_progress_bar=True, normalize_embeddings=True)
    sim = cosine_similarity(hi_emb, en_emb)
    return [
        {
            "hindi": hi,
            "candidates": [
                (english_sents[j], float(sim[i][j]))
                for j in np.argsort(sim[i])[::-1][:top_k]
            ]
        }
        for i, hi in enumerate(hindi_sents)
    ]

alignments = get_sentence_alignments(hindi_sentences, english_sentences)
print(f" Alignments computed: {len(alignments)}")

---
## STEP 13 — Word-by-Word Alignment

In [ ]:
def align_words(hindi_sent, english_sent, threshold=0.5):
    hi_tok = hindi_sent.split()
    en_tok = english_sent.split()
    if not hi_tok or not en_tok:
        return [], hi_tok
    hi_emb = model.encode(hi_tok, normalize_embeddings=True)
    en_emb = model.encode(en_tok, normalize_embeddings=True)
    sim    = cosine_similarity(hi_emb, en_emb)
    pairs, unaligned = [], []
    for i, hw in enumerate(hi_tok):
        best_j = int(np.argmax(sim[i]))
        score  = float(sim[i][best_j])
        if score >= threshold:
            pairs.append((hw, en_tok[best_j], round(score, 3)))
        else:
            unaligned.append(hw)
    return pairs, unaligned

print("--- Sample Word Alignments ---")
for aln in alignments[:3]:
    pairs, unaligned = align_words(aln["hindi"], aln["candidates"][0][0])
    print(f"\n HI: {aln['hindi']}")
    print(f" EN: {aln['candidates'][0][0]}")
    for hw, ew, sc in pairs:
        print(f"    {hw:20} ↔  {ew:20}  ({sc})")
    if unaligned:
        print(f"    Unaligned: {unaligned}")

---
## STEP 14 — POTATO Annotation JSONL

**Mode A** — Sentence alignment: annotator selects best English match for each Hindi sentence.  
**Mode B** — Word alignment: annotator selects best English token for each Hindi token.

In [ ]:
def create_sentence_alignment_jsonl(alignments, path="potato_sentence_align.jsonl"):
    records = []
    for i, aln in enumerate(alignments):
        r = {"id": f"sent_{i:04d}", "hindi_sentence": aln["hindi"]}
        for rank, (cand, score) in enumerate(aln["candidates"], 1):
            r[f"option_{rank}"] = cand
            r[f"score_{rank}"]  = round(score, 4)
        records.append(r)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f" [Mode A] {len(records)} sentence records → {path}")


def create_word_alignment_jsonl(alignments, path="potato_word_align.jsonl", top_k=4):
    records = []
    for i, aln in enumerate(alignments):
        hi, en   = aln["hindi"], aln["candidates"][0][0]
        hi_tok   = hi.split()
        en_tok   = en.split()
        if not hi_tok or not en_tok:
            continue
        hi_emb   = model.encode(hi_tok, normalize_embeddings=True)
        en_emb   = model.encode(en_tok, normalize_embeddings=True)
        sim      = cosine_similarity(hi_emb, en_emb)
        token_data = []
        for ti, hw in enumerate(hi_tok):
            top_idx = np.argsort(sim[ti])[::-1][:top_k]
            cands   = [
                {"english": en_tok[j], "score": round(float(sim[ti][j]), 3)}
                for j in top_idx
            ]
            cands.append({"english": "__NONE__", "score": 0.0})
            token_data.append({"hindi_token": hw, "candidates": cands})
        records.append({
            "id":               f"word_{i:04d}",
            "hindi_sentence":   hi,
            "english_sentence": en,
            "token_alignments": token_data
        })
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f" [Mode B] {len(records)} word alignment records → {path}")


create_sentence_alignment_jsonl(alignments)
create_word_alignment_jsonl(alignments)